# Building a Predictive Medical Diagnostic System
## Advanced ML Pipeline: Dimensionality Reduction · Feature Engineering · Multi-Class Classification
**Dataset:** 132 binary symptoms → 41 disease categories | **After dedup:** 304 rows × 132 features  
**Key challenges:** 133-feature multicollinearity · 41-class multinomial target · tiny test set (42 rows)


## 1. Library Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Core ML ───────────────────────────────────────────────────────────────────
from sklearn.preprocessing         import LabelEncoder, StandardScaler
from sklearn.model_selection       import (StratifiedKFold, RandomizedSearchCV,
                                            cross_val_score, StratifiedShuffleSplit)
from sklearn.metrics               import (accuracy_score, precision_score,
                                            recall_score, f1_score, roc_auc_score,
                                            classification_report, confusion_matrix,
                                            ConfusionMatrixDisplay)
from sklearn.utils.class_weight    import compute_class_weight
from sklearn.pipeline              import Pipeline

# ── Dimensionality Reduction ──────────────────────────────────────────────────
from sklearn.decomposition         import PCA, TruncatedSVD, NMF
from sklearn.manifold              import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.feature_selection     import (SelectKBest, chi2, mutual_info_classif,
                                            RFE, SelectFromModel, VarianceThreshold)

# ── Ensemble Models ───────────────────────────────────────────────────────────
from sklearn.ensemble              import (RandomForestClassifier, GradientBoostingClassifier,
                                            ExtraTreesClassifier, VotingClassifier,
                                            AdaBoostClassifier, BaggingClassifier,
                                            StackingClassifier)
from sklearn.tree                  import DecisionTreeClassifier
from sklearn.linear_model          import LogisticRegression
from sklearn.svm                   import SVC
import xgboost  as xgb
import lightgbm as lgb
from catboost                      import CatBoostClassifier

# ── Multicollinearity ─────────────────────────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.cluster.hierarchy             import linkage, fcluster, dendrogram
from scipy.spatial.distance              import squareform

# ── Stats ─────────────────────────────────────────────────────────────────────
from scipy.stats import chi2_contingency

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
RANDOM_STATE = 42
print("✅ All libraries imported successfully")


## 2. Data Loading & Initial Inspection

In [ ]:
train_df = pd.read_csv("DiseaseTraining.csv")
test_df  = pd.read_csv("DiseaseTesting.csv")

train_df = train_df.drop(columns=["Unnamed: 133"], errors='ignore')
test_df  = test_df.drop(columns=["Unnamed: 133"], errors='ignore')

print(f"Raw train shape  : {train_df.shape}")
print(f"Raw test shape   : {test_df.shape}")
print(f"Missing — train  : {train_df.isna().sum().sum()}")
print(f"Missing — test   : {test_df.isna().sum().sum()}")
print(f"Duplicates train : {train_df.duplicated().sum()}")
print(f"Target column    : prognosis | Unique classes: {train_df['prognosis'].nunique()}")
print(f"Feature dtype    : all binary (0/1) — {train_df.drop(columns=['prognosis']).nunique().max()} unique max")


## 3. Exploratory Data Analysis

### 3.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 6))
fig.suptitle("Disease Class Distribution — Training Data (Pre Dedup)", fontsize=14, fontweight='bold')

class_counts = train_df["prognosis"].value_counts()
palette = sns.color_palette("husl", len(class_counts))

axes[0].barh(class_counts.index, class_counts.values, color=palette)
axes[0].set_xlabel("Count", fontsize=11)
axes[0].set_title("Sample counts per disease (balanced: all = 120)", fontsize=11)
axes[0].axvline(class_counts.mean(), color='red', linestyle='--', label=f'Mean={class_counts.mean():.0f}')
axes[0].legend()

axes[1].pie(class_counts.values, labels=None, autopct='%1.1f%%',
            colors=palette, pctdistance=0.85, textprops={'fontsize':6})
axes[1].set_title("Proportional distribution — perfectly balanced ✓", fontsize=11)

plt.tight_layout()
plt.show()

print(f"Class balance ratio (max/min): {class_counts.max()/class_counts.min():.2f}")
print(f"→ Dataset is PERFECTLY balanced pre-dedup: {class_counts.nunique()==1}")


### 3.2 Symptom Prevalence & Co-occurrence

In [ ]:
# Drop duplicates for all remaining analysis
train_df = train_df.drop_duplicates().reset_index(drop=True)
train_df["prognosis"] = train_df["prognosis"].str.strip()
test_df["prognosis"]  = test_df["prognosis"].str.strip()
sym_cols = [c for c in train_df.columns if c != 'prognosis']

print(f"Post-dedup train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Top 30 symptoms by prevalence
symptom_prev = train_df[sym_cols].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
fig.suptitle("Symptom Analysis", fontsize=14, fontweight='bold')

# Top 30
axes[0].barh(symptom_prev.head(30).index[::-1], symptom_prev.head(30).values[::-1],
             color=sns.color_palette("Blues_r", 30))
axes[0].set_title("Top 30 Most Prevalent Symptoms", fontsize=11)
axes[0].set_xlabel("Occurrences across 304 rows")

# Distribution of symptom counts per row
row_counts = train_df[sym_cols].sum(axis=1)
axes[1].hist(row_counts, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(row_counts.mean(), color='red', linestyle='--', label=f'Mean={row_counts.mean():.1f}')
axes[1].set_title("Symptoms per Patient Row", fontsize=11)
axes[1].set_xlabel("# Active Symptoms"); axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Avg symptoms per patient: {row_counts.mean():.2f} | Range: {row_counts.min()}–{row_counts.max()}")


### 3.3 Symptom Correlation Heatmap (Top 40 Most Frequent)

In [ ]:
top40 = symptom_prev.head(40).index.tolist()
corr_mat = train_df[top40].corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.3, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.6, 'label': 'Pearson r'})
ax.set_title("Symptom Correlation Matrix (Top 40 Symptoms) — reveals multicollinearity clusters",
             fontsize=12, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout(); plt.show()

# Report high correlations
full_corr = train_df[sym_cols].corr().abs()
upper_tri = full_corr.where(np.triu(np.ones(full_corr.shape), k=1).astype(bool))
high_corr_pairs = upper_tri.stack()[upper_tri.stack() > 0.85]
print(f"⚠️  Highly correlated pairs (|r| > 0.85): {len(high_corr_pairs)}")
print("\nTop 10 most correlated pairs:")
print(high_corr_pairs.sort_values(ascending=False).head(10).to_string())


### 3.4 Disease–Symptom Heatmap (Disease Profiles)

In [ ]:
disease_profiles = train_df.groupby('prognosis')[sym_cols].mean()

# Find top 40 most discriminative symptoms (highest variance across diseases)
disc_syms = disease_profiles.var().sort_values(ascending=False).head(40).index.tolist()

fig, ax = plt.subplots(figsize=(22, 12))
sns.heatmap(disease_profiles[disc_syms].T, cmap='YlOrRd',
            xticklabels=True, yticklabels=True,
            ax=ax, linewidths=0.2,
            cbar_kws={'label': 'Mean symptom presence', 'shrink': 0.4})
ax.set_title("Disease × Symptom Profile Matrix (Top 40 Discriminative Symptoms)",
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout(); plt.show()


### 3.5 PCA Scree Plot — Understanding Intrinsic Dimensionality

In [ ]:
from sklearn.preprocessing import StandardScaler

X_raw = train_df[sym_cols].values.astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n90, n95, n99 = (np.argmax(cumvar >= t) + 1 for t in [0.90, 0.95, 0.99])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(range(1, len(cumvar)+1), cumvar, 'b-', lw=1.5)
axes[0].fill_between(range(1, len(cumvar)+1), cumvar, alpha=0.15)
for n, t, c in [(n90, 0.90, 'red'), (n95, 0.95, 'orange'), (n99, 0.99, 'green')]:
    axes[0].axhline(t, color=c, linestyle='--', lw=1)
    axes[0].axvline(n, color=c, linestyle=':', lw=1, label=f'{int(t*100)}% @ PC{n}')
axes[0].set_xlabel("Number of Principal Components")
axes[0].set_ylabel("Cumulative Explained Variance")
axes[0].set_title("PCA Scree — Cumulative Variance", fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].bar(range(1, 51), pca_full.explained_variance_ratio_[:50],
            color='steelblue', alpha=0.8)
axes[1].set_xlabel("Principal Component"); axes[1].set_ylabel("Explained Variance Ratio")
axes[1].set_title("Individual PC Variance (First 50)", fontsize=11, fontweight='bold')

plt.suptitle(f"PCA Analysis: 132 features → {n95} PCs explain 95% variance", fontsize=12)
plt.tight_layout(); plt.show()

print(f"Components for 90% variance: {n90}")
print(f"Components for 95% variance: {n95}")
print(f"Components for 99% variance: {n99}")
print(f"Dimensionality reduction ratio (95%): {n95/132:.1%} of original")


## 4. Advanced Data Cleaning & Feature Engineering
### 4.1 Multicollinearity Treatment — Hierarchical Clustering of Features
Binary features with high Pearson correlation are grouped via hierarchical clustering.
One representative is kept per cluster (the most informative one), reducing redundancy
without losing discriminative power.


In [ ]:
# ── Step 1: Variance threshold — remove near-zero variance ──────────────────
vt = VarianceThreshold(threshold=0.005)
vt.fit(X_raw)
selected_by_var = np.array(sym_cols)[vt.get_support()].tolist()
removed_by_var  = set(sym_cols) - set(selected_by_var)
print(f"Removed by variance threshold (<0.5%): {removed_by_var}")
print(f"Features retained: {len(selected_by_var)}")

X_var = train_df[selected_by_var].values.astype(float)

# ── Step 2: Hierarchical clustering to handle multicollinearity ───────────────
# Convert correlation matrix to distance matrix → cluster → keep one per cluster
corr_full = pd.DataFrame(X_var, columns=selected_by_var).corr().abs()
corr_full = corr_full.fillna(0)
dist_matrix = 1 - corr_full.values
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.clip(dist_matrix, 0, None)

condensed_dist = squareform(dist_matrix)
Z = linkage(condensed_dist, method='complete')

# Cut tree at distance threshold = 0.15 (keeps features with |r| < 0.85)
DIST_THRESHOLD = 0.15
cluster_labels = fcluster(Z, t=DIST_THRESHOLD, criterion='distance')
n_clusters     = len(np.unique(cluster_labels))
print(f"\nHierarchical clustering at distance {DIST_THRESHOLD}:")
print(f"  {len(selected_by_var)} features → {n_clusters} clusters")

# Keep the feature with highest mutual info per cluster
from sklearn.preprocessing import LabelEncoder
le_temp = LabelEncoder()
y_temp  = le_temp.fit_transform(train_df['prognosis'].values)

mi_scores = mutual_info_classif(X_var, y_temp, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=selected_by_var)

kept_features_hc = []
cluster_map      = {}
for c in np.unique(cluster_labels):
    feat_in_cluster = [selected_by_var[i] for i, cl in enumerate(cluster_labels) if cl == c]
    best_feat = mi_series[feat_in_cluster].idxmax()
    kept_features_hc.append(best_feat)
    cluster_map[c] = (feat_in_cluster, best_feat)

print(f"  Features after clustering (one per cluster): {len(kept_features_hc)}")

# Show a few clusters
print("\nSample clusters (redundant features grouped together):")
for c, (feats, best) in list(cluster_map.items())[:5]:
    if len(feats) > 1:
        print(f"  Cluster {c}: {feats} → kept '{best}'")


### 4.2 VIF-Based Multicollinearity Removal (Iterative)

In [ ]:
def compute_vif_df(X_df):
    '''Compute VIF for all columns in a DataFrame.'''
    vif = pd.DataFrame()
    vif['feature'] = X_df.columns
    vif['VIF']     = [variance_inflation_factor(X_df.values.astype(float), i)
                      for i in range(X_df.shape[1])]
    return vif.sort_values('VIF', ascending=False).reset_index(drop=True)

def iterative_vif_elimination(X_df, threshold=10.0, max_iter=50):
    '''
    Iteratively removes the highest-VIF feature until all VIF < threshold.
    Threshold=10 is standard; lower (5) is conservative.
    '''
    cols   = list(X_df.columns)
    removed = []
    for iteration in range(max_iter):
        vif_df = compute_vif_df(X_df[cols])
        max_vif_row = vif_df.iloc[0]
        if max_vif_row['VIF'] <= threshold:
            break
        drop_col = max_vif_row['feature']
        cols.remove(drop_col)
        removed.append((drop_col, max_vif_row['VIF']))
        if (iteration+1) % 5 == 0:
            print(f"  Iteration {iteration+1}: removed '{drop_col}' (VIF={max_vif_row['VIF']:.1f}), {len(cols)} remain")
    return cols, removed

print("Running iterative VIF elimination (threshold=10)...")
print("This may take ~1 minute with 100+ binary features...\n")

# Use hc-selected features to reduce computation
hc_df = train_df[kept_features_hc].copy()
vif_kept_cols, vif_removed = iterative_vif_elimination(hc_df, threshold=10.0)

print(f"\n✅ VIF elimination complete:")
print(f"  Input features  : {len(kept_features_hc)}")
print(f"  Removed by VIF  : {len(vif_removed)}")
print(f"  Remaining        : {len(vif_kept_cols)}")
print(f"\nFinal feature set for modelling: {len(vif_kept_cols)} features (from original {len(sym_cols)})")


### 4.3 Domain-Driven Feature Engineering (Symptom Grouping)

In [ ]:
# ── Clinically meaningful symptom groups ─────────────────────────────────────
SYMPTOM_GROUPS = {
    'gastro_score':      ['nausea','vomiting','stomach_pain','diarrhoea','indigestion',
                          'loss_of_appetite','stomach_bleeding','distention_of_abdomen',
                          'acidity','constipation'],
    'respiratory_score': ['breathlessness','cough','phlegm','throat_irritation',
                          'rusty_sputum','blood_in_sputum','mucoid_sputum','continuous_sneezing'],
    'neuro_score':       ['headache','dizziness','unsteadiness','loss_of_balance',
                          'altered_sensorium','slurred_speech','spinning_movements',
                          'visual_disturbances','loss_of_smell'],
    'skin_score':        ['itching','skin_rash','nodal_skin_eruptions','dischromic_patches',
                          'yellowish_skin','skin_peeling','silver_like_dusting','blister',
                          'pus_filled_pimples','blackheads','red_spots_over_body'],
    'liver_score':       ['yellowing_of_eyes','dark_urine','yellowish_skin','bile_vomit',
                          'stomach_bleeding','distention_of_abdomen','fluid_overload'],
    'musculo_score':     ['joint_pain','knee_pain','hip_joint_pain','muscle_weakness',
                          'back_pain','neck_pain','muscle_wasting','stiff_neck',
                          'swollen_joints','movement_stiffness','painful_walking'],
    'fever_score':       ['high_fever','mild_fever','sweating','chills','shivering',
                          'malaise','fatigue'],
    'cardiac_score':     ['chest_pain','palpitations','fast_heart_rate','breathlessness',
                          'sweating','cold_hands_and_feets'],
    'metabolic_score':   ['weight_gain','weight_loss','obesity','excessive_hunger',
                          'increased_appetite','polyuria','irregular_sugar_level'],
    'infection_score':   ['high_fever','swelled_lymph_nodes','malaise','fatigue',
                          'red_spots_over_body','redness_of_eyes','runny_nose'],
}

def add_group_scores(df, sym_cols_present):
    '''Add symptom group aggregate scores as new engineered features.'''
    df = df.copy()
    for group_name, symptoms in SYMPTOM_GROUPS.items():
        available = [s for s in symptoms if s in sym_cols_present]
        if available:
            df[group_name] = df[available].sum(axis=1)
    return df

train_fe = add_group_scores(train_df, sym_cols)
test_fe  = add_group_scores(test_df,  sym_cols)

eng_cols = list(SYMPTOM_GROUPS.keys())
print("Engineered symptom group scores:")
for c in eng_cols:
    print(f"  {c:25s}: range {train_fe[c].min():.0f}–{train_fe[c].max():.0f}, mean={train_fe[c].mean():.2f}")


### 4.4 Interaction & Statistical Features

In [ ]:
# ── Feature 1: Total symptom burden (count) ──────────────────────────────────
train_fe['total_symptoms'] = train_fe[sym_cols].sum(axis=1)
test_fe['total_symptoms']  = test_fe[sym_cols].sum(axis=1)

# ── Feature 2: Symptom rarity score (sum of inverse-frequency weights) ────────
# Rare symptoms that appear carry more diagnostic signal
symptom_freq  = train_df[sym_cols].mean()   # prevalence per feature
rarity_weight = 1.0 / (symptom_freq + 1e-6)
rarity_weight = rarity_weight / rarity_weight.sum()   # normalise

train_fe['rarity_score'] = train_fe[sym_cols].values @ rarity_weight.values
test_fe['rarity_score']  = test_fe[sym_cols].values  @ rarity_weight.values

# ── Feature 3: Symptom entropy (uncertainty within a patient row) ─────────────
# High entropy → symptoms spread across many systems; low → concentrated
def symptom_entropy(row, cols):
    probs = row[cols].values.astype(float)
    probs = probs / (probs.sum() + 1e-9)
    return -np.sum(probs * np.log2(probs + 1e-9))

train_fe['symptom_entropy'] = train_fe.apply(symptom_entropy, cols=sym_cols, axis=1)
test_fe['symptom_entropy']  = test_fe.apply(symptom_entropy,  cols=sym_cols, axis=1)

# ── Feature 4: Multi-system involvement (# of groups with ≥1 symptom) ────────
for name, symptoms in SYMPTOM_GROUPS.items():
    avail = [s for s in symptoms if s in sym_cols]
    train_fe[f'has_{name}'] = (train_fe[avail].sum(axis=1) > 0).astype(int)
    test_fe[f'has_{name}']  = (test_fe[avail].sum(axis=1)  > 0).astype(int)

has_cols = [f'has_{n}' for n in SYMPTOM_GROUPS.keys()]
train_fe['system_breadth'] = train_fe[has_cols].sum(axis=1)
test_fe['system_breadth']  = test_fe[has_cols].sum(axis=1)

# ── Feature 5: Dominant system (argmax of group scores) ──────────────────────
dominant_sys = train_fe[eng_cols].idxmax(axis=1)
dominant_sys_num = LabelEncoder().fit_transform(dominant_sys)
train_fe['dominant_system'] = dominant_sys_num

dominant_sys_t = test_fe[eng_cols].idxmax(axis=1)
le_dom = LabelEncoder().fit(dominant_sys)
test_fe['dominant_system'] = le_dom.transform(dominant_sys_t)

# ── Feature 6: Chi-squared top features (filter method) ──────────────────────
le_chi = LabelEncoder()
y_chi  = le_chi.fit_transform(train_fe['prognosis'])
chi_sel = SelectKBest(chi2, k=60)
chi_sel.fit(train_fe[sym_cols].values, y_chi)
chi_selected_feats = np.array(sym_cols)[chi_sel.get_support()].tolist()

# ── Feature 7: Mutual information top features (non-linear filter) ───────────
mi_sel = SelectKBest(mutual_info_classif, k=60)
mi_sel.fit(train_fe[sym_cols].values, y_chi)
mi_selected_feats  = np.array(sym_cols)[mi_sel.get_support()].tolist()

union_selected = list(set(chi_selected_feats) | set(mi_selected_feats))
print(f"Chi² selected  : {len(chi_selected_feats)} features")
print(f"MI  selected   : {len(mi_selected_feats)} features")
print(f"Union selection: {len(union_selected)} features")

extra_eng_cols = eng_cols + has_cols + ['total_symptoms','rarity_score',
                                         'symptom_entropy','system_breadth',
                                         'dominant_system']
print(f"\nTotal engineered features added: {len(extra_eng_cols)}")


## 5. Dimensionality Reduction Suite
Five complementary techniques — each has different assumptions.
We'll compare their effect on downstream classification.


In [ ]:
# ── Prepare base feature matrix ───────────────────────────────────────────────
le_main = LabelEncoder()
y_train = le_main.fit_transform(train_fe['prognosis'].values)
y_test  = le_main.transform(test_fe['prognosis'].values)

# Use VIF-filtered + engineered features as base
base_feat_cols = vif_kept_cols + extra_eng_cols
base_feat_cols = [c for c in base_feat_cols if c in train_fe.columns]

X_train_base = train_fe[base_feat_cols].values.astype(float)
X_test_base  = test_fe[base_feat_cols].values.astype(float)

scaler_base = StandardScaler()
X_train_sc  = scaler_base.fit_transform(X_train_base)
X_test_sc   = scaler_base.transform(X_test_base)

print(f"Base feature matrix: {X_train_sc.shape[1]} features")
print(f"  (from original 132 → VIF-filtered → + engineered)")

# ── 1. PCA ─────────────────────────────────────────────────────────────────────
pca_95 = PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)
X_train_pca = pca_95.fit_transform(X_train_sc)
X_test_pca  = pca_95.transform(X_test_sc)
print(f"\n[1] PCA (95% var): {X_train_pca.shape[1]} components")

# ── 2. LDA (supervised — best for multinomial classification) ─────────────────
# LDA max components = min(n_classes-1, n_features) = min(40, base_cols)
lda = LDA(solver='svd', n_components=min(40, X_train_sc.shape[1]-1))
X_train_lda = lda.fit_transform(X_train_sc, y_train)
X_test_lda  = lda.transform(X_test_sc)
print(f"[2] LDA (supervised): {X_train_lda.shape[1]} discriminant components")
print(f"    → LDA is IDEAL for 41-class problem: maximises class separation")

# ── 3. Truncated SVD (works directly on sparse binary matrices) ───────────────
# Great for binary data — doesn't require centering, avoids dense memory
X_sparse_train = train_fe[sym_cols].values.astype(float)  # raw binary
X_sparse_test  = test_fe[sym_cols].values.astype(float)

tsvd = TruncatedSVD(n_components=50, random_state=RANDOM_STATE)
X_train_svd = tsvd.fit_transform(X_sparse_train)
X_test_svd  = tsvd.transform(X_sparse_test)
print(f"[3] TruncatedSVD: {X_train_svd.shape[1]} components from sparse binary matrix")

# ── 4. Combined: LDA features + engineered features ───────────────────────────
# This hybrid captures both linear discrimination AND domain knowledge
eng_train = train_fe[extra_eng_cols].values.astype(float)
eng_test  = test_fe[extra_eng_cols].values.astype(float)

scaler_eng = StandardScaler()
eng_train_sc = scaler_eng.fit_transform(eng_train)
eng_test_sc  = scaler_eng.transform(eng_test)

X_train_hybrid = np.hstack([X_train_lda, eng_train_sc])
X_test_hybrid  = np.hstack([X_test_lda,  eng_test_sc])
print(f"[4] Hybrid (LDA + Engineered): {X_train_hybrid.shape[1]} features")

# ── 5. t-SNE for visualisation (not for modelling — non-parametric) ───────────
print("\n[5] Computing t-SNE for visualisation (2D)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, n_iter=1000)
X_tsne = tsne.fit_transform(X_train_lda)   # use LDA output for speed
print(f"    t-SNE output: {X_tsne.shape} (visualisation only)")


In [ ]:
# ── t-SNE visualisation ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle("t-SNE Projection of Training Data via LDA Features", fontsize=13, fontweight='bold')

unique_diseases = train_fe['prognosis'].unique()
palette_tsne = dict(zip(unique_diseases, 
                        sns.color_palette("gist_rainbow", len(unique_diseases))))

for ax_idx, ax in enumerate(axes):
    for disease in unique_diseases:
        mask = (train_fe['prognosis'].values == disease)
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                   c=[palette_tsne[disease]], label=disease, s=35, alpha=0.75)
    ax.set_title("t-SNE — colour by disease (LDA compressed)", fontsize=10)
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
    if ax_idx == 1:
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
                  fontsize=6, ncol=2, borderaxespad=0)

plt.tight_layout()
plt.show()
print("→ Well-separated clusters indicate strong discriminative power of LDA features")


In [ ]:
# ── Dimensionality reduction comparison summary ────────────────────────────────
dr_summary = pd.DataFrame({
    'Method':        ['Raw Binary',  'PCA (95%)',  'LDA',        'TruncatedSVD', 'Hybrid (LDA+Eng)'],
    'Features':      [132,           X_train_pca.shape[1], X_train_lda.shape[1],
                      X_train_svd.shape[1], X_train_hybrid.shape[1]],
    'Supervised':    ['No',          'No',         'Yes',         'No',           'Mixed'],
    'Best for':      ['Baseline',    'Unsupervised', '41-class', 'Sparse binary', 'Combined'],
    'Reduction %':   [0,
                      round((1 - X_train_pca.shape[1]/132)*100, 1),
                      round((1 - X_train_lda.shape[1]/132)*100, 1),
                      round((1 - X_train_svd.shape[1]/132)*100, 1),
                      round((1 - X_train_hybrid.shape[1]/132)*100, 1)],
})
print("\n📊 Dimensionality Reduction Summary:")
print(dr_summary.to_string(index=False))
print("\n💡 LDA is theoretically optimal for multinomial classification:")
print("   It projects to a space that MAXIMISES between-class variance")
print("   and MINIMISES within-class variance — exactly what we need for 41 diseases.")


## 6. Handling the 41-Class Multinomial Problem
### Strategy Discussion
With 41 classes and only 304 rows (≈7.4 rows/class), we face:
- **High-dimensional** decision boundary (each class needs enough support)
- **Multinomial log-loss** as optimisation target (not binary cross-entropy)
- **OvR vs OvO** strategy for models that require it (SVM, LR)
- **Macro vs weighted** metrics — both matter


In [ ]:
# ── Class weight computation ────────────────────────────────────────────────────
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(zip(np.unique(y_train), class_weights_arr))

print("Class balancing strategy: class_weight='balanced'")
print(f"Number of unique classes : {len(class_weight_dict)}")
print(f"Weight range             : {min(class_weights_arr):.3f} – {max(class_weights_arr):.3f}")
print(f"All weights equal        : {np.allclose(class_weights_arr, class_weights_arr[0])}")
print("→ Post-dedup dataset is highly IMBALANCED (7–8 samples/class)")
print("  class_weight='balanced' compensates by scaling loss per class")

# Sample weight vector for algorithms that take sample_weight
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_train = compute_sample_weight('balanced', y_train)
print(f"\nSample weight range: {sample_weights_train.min():.3f} – {sample_weights_train.max():.3f}")

# Stratified cross-validation is CRITICAL with 41 classes and tiny dataset
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"\nStratified 5-fold CV: ensures each fold has ~{len(y_train)//5} rows")
print(f"  → Each class appears in every fold (stratification = essential here)")


## 7. Model Evaluation Framework

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, le):
    '''
    Comprehensive evaluation for 41-class multinomial problem.
    Reports accuracy, macro/weighted precision/recall/F1, ROC-AUC (OvR macro).
    '''
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1w  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1m  = f1_score(y_test, y_pred, average='macro',    zero_division=0)
    
    roc  = np.nan
    if y_proba is not None:
        try:
            roc = roc_auc_score(y_test, y_proba, multi_class='ovr',
                                average='macro')
        except Exception:
            pass
    
    result = {
        'Model':     model_name,
        'Accuracy':  round(acc,  4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
        'F1_W':      round(f1w,  4),
        'F1_Macro':  round(f1m,  4),
        'ROC_AUC':   round(roc,  4) if not np.isnan(roc) else np.nan,
    }
    return result

def cv_evaluate(model, X, y, cv, scoring='f1_weighted', name=''):
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"  [{name}] CV {scoring}: {scores.round(4)} | mean={scores.mean():.4f} ± {scores.std():.4f}")
    return scores

print("✅ Evaluation framework ready")
print("   Metrics: Accuracy · Precision(W) · Recall(W) · F1(Weighted) · F1(Macro) · ROC-AUC(OvR Macro)")
print("   CV: StratifiedKFold(5) — stratification critical for 41-class problem")


## 8. Model 1 — Random Forest (with Exhaustive Tuning)
Training on **Raw (132 features)** — RF handles multicollinearity via random feature sampling.


In [ ]:
print("=" * 60)
print("MODEL 1: Random Forest Classifier")
print("=" * 60)

rf_param_grid = {
    "n_estimators"      : [200, 400, 600, 800],
    "max_depth"         : [None, 15, 25, 40],
    "min_samples_split" : [2, 4, 6],
    "min_samples_leaf"  : [1, 2],
    "max_features"      : ['sqrt', 'log2', 0.3],
    "bootstrap"         : [True, False],
    "criterion"         : ['gini', 'entropy'],
}

X_train_rf = train_fe[sym_cols + extra_eng_cols].values.astype(float)
X_test_rf  = test_fe[sym_cols + extra_eng_cols].values.astype(float)

rf_base = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, rf_param_grid, n_iter=60,
    scoring='f1_weighted', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
rf_search.fit(X_train_rf, y_train)
best_rf = rf_search.best_estimator_

print(f"\nBest RF params:")
for k, v in rf_search.best_params_.items():
    print(f"  {k:25s}: {v}")
print(f"Best CV F1 (weighted): {rf_search.best_score_:.4f}")

rf_cv = cv_evaluate(best_rf, X_train_rf, y_train, cv_strat, 'f1_weighted', 'RF')
rf_results = evaluate_model(best_rf, X_test_rf, y_test, "Random Forest", le_main)
print(f"\n✅ RF Test Results: {rf_results}")


## 9. Model 2 — XGBoost with Multinomial Objective

In [ ]:
print("=" * 60)
print("MODEL 2: XGBoost — multinomial:softmax")
print("=" * 60)

# Train on LDA-transformed features (supervised DR = best for multinomial)
X_train_xgb = X_train_lda
X_test_xgb  = X_test_lda

xgb_param_grid = {
    "n_estimators"         : [200, 400, 600],
    "max_depth"            : [3, 5, 7, 9],
    "learning_rate"        : [0.01, 0.05, 0.1, 0.2],
    "subsample"            : [0.6, 0.8, 1.0],
    "colsample_bytree"     : [0.6, 0.8, 1.0],
    "colsample_bylevel"    : [0.6, 0.8],
    "reg_alpha"            : [0, 0.1, 0.5, 1.0],
    "reg_lambda"           : [0.5, 1.0, 2.0],
    "min_child_weight"     : [1, 3, 5],
    "gamma"                : [0, 0.1, 0.3],
}

xgb_base = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(np.unique(y_train)),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
xgb_search = RandomizedSearchCV(
    xgb_base, xgb_param_grid, n_iter=60,
    scoring='f1_weighted', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train_xgb, y_train, sample_weight=sample_weights_train)
best_xgb = xgb_search.best_estimator_

print(f"\nBest XGBoost params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k:25s}: {v}")
print(f"Best CV F1 (weighted): {xgb_search.best_score_:.4f}")

xgb_cv = cv_evaluate(best_xgb, X_train_xgb, y_train, cv_strat, 'f1_weighted', 'XGBoost')
xgb_results = evaluate_model(best_xgb, X_test_xgb, y_test, "XGBoost (LDA)", le_main)
print(f"\n✅ XGBoost Test Results: {xgb_results}")


## 10. Model 3 — LightGBM (native multiclass, fast)

In [ ]:
print("=" * 60)
print("MODEL 3: LightGBM — multiclass")
print("=" * 60)

# Use Hybrid features for LightGBM
X_train_lgb = X_train_hybrid
X_test_lgb  = X_test_hybrid

lgb_param_grid = {
    "n_estimators"      : [200, 400, 600, 800],
    "max_depth"         : [-1, 8, 15, 25],
    "learning_rate"     : [0.005, 0.01, 0.05, 0.1],
    "num_leaves"        : [15, 31, 63, 127],
    "reg_alpha"         : [0, 0.1, 0.5],
    "reg_lambda"        : [0, 0.1, 1.0, 5.0],
    "colsample_bytree"  : [0.6, 0.8, 1.0],
    "subsample"         : [0.6, 0.8, 1.0],
    "min_child_samples" : [5, 10, 20],
    "min_child_weight"  : [0.001, 0.01, 0.1],
}

lgb_base = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(np.unique(y_train)),
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgb_search = RandomizedSearchCV(
    lgb_base, lgb_param_grid, n_iter=60,
    scoring='f1_weighted', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
lgb_search.fit(X_train_lgb, y_train)
best_lgb = lgb_search.best_estimator_

print(f"\nBest LightGBM params:")
for k, v in lgb_search.best_params_.items():
    print(f"  {k:25s}: {v}")
print(f"Best CV F1 (weighted): {lgb_search.best_score_:.4f}")

lgb_cv = cv_evaluate(best_lgb, X_train_lgb, y_train, cv_strat, 'f1_weighted', 'LightGBM')
lgb_results = evaluate_model(best_lgb, X_test_lgb, y_test, "LightGBM (Hybrid)", le_main)
print(f"\n✅ LightGBM Test Results: {lgb_results}")


## 11. Model 4 — CatBoost (handles multinomial natively, no scaling needed)

In [ ]:
print("=" * 60)
print("MODEL 4: CatBoost — MultiClass")
print("=" * 60)

# CatBoost trains on raw features + engineered (no need for dimensionality reduction)
X_train_cat = train_fe[sym_cols + extra_eng_cols].values.astype(float)
X_test_cat  = test_fe[sym_cols + extra_eng_cols].values.astype(float)

cat_param_grid = {
    "iterations"        : [300, 500, 800],
    "learning_rate"     : [0.01, 0.05, 0.1, 0.2],
    "depth"             : [4, 6, 8, 10],
    "l2_leaf_reg"       : [1, 3, 5, 10],
    "border_count"      : [32, 64, 128],
    "bagging_temperature": [0, 0.5, 1.0],
    "random_strength"   : [0, 0.5, 1.0],
}

cat_base = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='Accuracy',
    auto_class_weights='Balanced',
    random_state=RANDOM_STATE,
    verbose=0,
    thread_count=-1,
)
cat_search = RandomizedSearchCV(
    cat_base, cat_param_grid, n_iter=40,
    scoring='f1_weighted', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=1, verbose=1   # CatBoost is already parallel
)
cat_search.fit(X_train_cat, y_train)
best_cat = cat_search.best_estimator_

print(f"\nBest CatBoost params:")
for k, v in cat_search.best_params_.items():
    print(f"  {k:25s}: {v}")
print(f"Best CV F1 (weighted): {cat_search.best_score_:.4f}")

cat_cv = cv_evaluate(best_cat, X_train_cat, y_train, cv_strat, 'f1_weighted', 'CatBoost')
cat_results = evaluate_model(best_cat, X_test_cat, y_test, "CatBoost", le_main)
print(f"\n✅ CatBoost Test Results: {cat_results}")


## 12. Model 5 — Extra Trees (Fast, Low-variance, Multiclass-native)

In [ ]:
print("=" * 60)
print("MODEL 5: Extra Trees — diversity via extreme randomness")
print("=" * 60)

et_param_grid = {
    "n_estimators"      : [200, 400, 600],
    "max_depth"         : [None, 20, 30],
    "min_samples_split" : [2, 4, 6],
    "min_samples_leaf"  : [1, 2],
    "max_features"      : ['sqrt', 'log2', 0.4],
    "criterion"         : ['gini', 'entropy'],
}

et_base = ExtraTreesClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
et_search = RandomizedSearchCV(
    et_base, et_param_grid, n_iter=40,
    scoring='f1_weighted', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
et_search.fit(X_train_rf, y_train)
best_et = et_search.best_estimator_

print(f"Best CV F1 (weighted): {et_search.best_score_:.4f}")

et_cv = cv_evaluate(best_et, X_train_rf, y_train, cv_strat, 'f1_weighted', 'ExtraTrees')
et_results = evaluate_model(best_et, X_test_rf, y_test, "Extra Trees", le_main)
print(f"\n✅ Extra Trees Test Results: {et_results}")


## 13. Model 6 — Soft Voting Ensemble (RF + XGB + LGB + CatBoost)

In [ ]:
print("=" * 60)
print("MODEL 6: Soft Voting Ensemble (OvR wrapper for unified features)")
print("=" * 60)
# Voting requires same feature space — use Hybrid (LDA + engineered)
from sklearn.pipeline import make_pipeline

rf_pipe  = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(**{k:v for k,v in best_rf.get_params().items()
                              if k != 'n_jobs'}, n_jobs=-1, random_state=RANDOM_STATE,
                           class_weight='balanced')
)
lgb_pipe = lgb.LGBMClassifier(**best_lgb.get_params(), verbose=-1)
cat_pipe = CatBoostClassifier(**{k:v for k,v in best_cat.get_params().items()},
                               verbose=0)

voting_clf = VotingClassifier(
    estimators=[
        ('rf',   rf_pipe),
        ('lgbm', lgb_pipe),
        ('cat',  cat_pipe),
    ],
    voting='soft',
    n_jobs=1,
)

# Use hybrid features (consistent for all estimators)
voting_clf.fit(X_train_hybrid, y_train)
print("✅ Soft Voting Classifier fitted")

voting_results = evaluate_model(voting_clf, X_test_hybrid, y_test,
                                "Voting (RF+LGB+Cat)", le_main)
print(f"Voting Test Results: {voting_results}")


## 14. Model 7 — Stacking Classifier (Meta-Learner)

In [ ]:
print("=" * 60)
print("MODEL 7: Stacking Classifier — Logistic Regression meta-learner")
print("=" * 60)

from sklearn.linear_model import LogisticRegression

# Level-0 estimators — diverse base learners
estimators_stack = [
    ('rf',   RandomForestClassifier(n_estimators=300, max_depth=20,
                                    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('et',   ExtraTreesClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)),
    ('lgbm', lgb.LGBMClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=RANDOM_STATE, verbose=-1)),
]

# Level-1 meta-learner — multinomial Logistic Regression
meta_lr = LogisticRegression(
    C=1.0, solver='lbfgs', max_iter=2000,
    multi_class='multinomial',
    class_weight='balanced',
    random_state=RANDOM_STATE
)

stack_clf = StackingClassifier(
    estimators=estimators_stack,
    final_estimator=meta_lr,
    cv=cv_strat,
    stack_method='predict_proba',
    n_jobs=1,
)

stack_clf.fit(X_train_rf, y_train)
print("✅ Stacking Classifier fitted")

stack_results = evaluate_model(stack_clf, X_test_rf, y_test,
                               "Stacking (RF+ET+LGB → LR)", le_main)
print(f"Stacking Test Results: {stack_results}")


## 15. Comprehensive Model Comparison

In [ ]:
all_results = [rf_results, xgb_results, lgb_results, cat_results,
               et_results, voting_results, stack_results]

results_df = pd.DataFrame(all_results).set_index('Model')
results_df_sorted = results_df.sort_values('F1_W', ascending=False)

print("\n" + "="*80)
print(f"{'FINAL MODEL COMPARISON — ALL METRICS':^80}")
print("="*80)
print(results_df_sorted.to_string())
print("="*80)

best_acc = results_df['Accuracy'].idxmax()
best_f1w = results_df['F1_W'].idxmax()
best_auc = results_df['ROC_AUC'].idxmax()
best_f1m = results_df['F1_Macro'].idxmax()

print(f"\n🏆 Best Accuracy  : {best_acc} ({results_df.loc[best_acc,'Accuracy']:.4f})")
print(f"🏆 Best F1 Weighted: {best_f1w} ({results_df.loc[best_f1w,'F1_W']:.4f})")
print(f"🏆 Best F1 Macro   : {best_f1m} ({results_df.loc[best_f1m,'F1_Macro']:.4f})")
print(f"🏆 Best ROC-AUC    : {best_auc} ({results_df.loc[best_auc,'ROC_AUC']:.4f})")


In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Model Performance Comparison — 41-Class Multinomial Problem",
             fontsize=14, fontweight='bold')

metrics  = ['Accuracy', 'F1_W', 'F1_Macro', 'ROC_AUC']
labels   = ['Accuracy', 'F1 Weighted', 'F1 Macro', 'ROC-AUC (OvR)']
palettes = ['Blues_d', 'Greens_d', 'Oranges_d', 'Purples_d']

for ax, metric, label, pal in zip(axes.flatten(), metrics, labels, palettes):
    vals  = results_df_sorted[metric].dropna()
    colors_bar = sns.color_palette(pal, len(vals))[::-1]
    bars  = ax.barh(vals.index, vals.values, color=colors_bar, edgecolor='white')
    ax.set_xlim(0, 1.05)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Score')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.axvline(0.9, color='red', linestyle='--', alpha=0.5, lw=1)

plt.tight_layout(); plt.show()


In [ ]:
# ── CV score comparison ────────────────────────────────────────────────────────
cv_scores_dict = {
    "Random Forest": rf_cv,
    "XGBoost":       xgb_cv,
    "LightGBM":      lgb_cv,
    "Extra Trees":   et_cv,
}

fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(cv_scores_dict))
colors_cv = sns.color_palette("Set2", len(cv_scores_dict))

for i, (name, scores) in enumerate(cv_scores_dict.items()):
    ax.boxplot(scores, positions=[i], widths=0.35,
               patch_artist=True,
               boxprops=dict(facecolor=colors_cv[i], alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
    ax.scatter([i]*len(scores), scores, color=colors_cv[i], zorder=5, s=40, alpha=0.9)
    ax.text(i, scores.mean() + 0.002, f'{scores.mean():.3f}',
            ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(positions)
ax.set_xticklabels(cv_scores_dict.keys(), fontsize=11)
ax.set_ylabel("F1 Weighted (5-Fold CV)")
ax.set_title("Cross-Validation Score Distributions — Stratified 5-Fold", fontsize=12, fontweight='bold')
ax.set_ylim(0.7, 1.05)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()


## 16. Feature Importance Analysis

In [ ]:
# ── RF and ET feature importance ──────────────────────────────────────────────
feat_names_rf = sym_cols + extra_eng_cols
rf_imp = pd.Series(best_rf.feature_importances_, index=feat_names_rf)
et_imp = pd.Series(best_et.feature_importances_, index=feat_names_rf)
avg_imp = ((rf_imp + et_imp) / 2).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(24, 9))
fig.suptitle("Feature Importance Analysis", fontsize=13, fontweight='bold')

# RF top 30
top30_rf = rf_imp.sort_values(ascending=False).head(30)
axes[0].barh(top30_rf.index[::-1], top30_rf.values[::-1],
             color=sns.color_palette("Blues_r", 30))
axes[0].set_title("Random Forest — Top 30 Features", fontsize=10)
axes[0].set_xlabel("Importance Score")

# ET top 30
top30_et = et_imp.sort_values(ascending=False).head(30)
axes[1].barh(top30_et.index[::-1], top30_et.values[::-1],
             color=sns.color_palette("Greens_r", 30))
axes[1].set_title("Extra Trees — Top 30 Features", fontsize=10)
axes[1].set_xlabel("Importance Score")

# Engineered vs original feature importance
eng_total   = avg_imp[[c for c in avg_imp.index if c in extra_eng_cols]].sum()
orig_total  = avg_imp[[c for c in avg_imp.index if c in sym_cols]].sum()
axes[2].pie([eng_total, orig_total],
            labels=[f'Engineered\n({len(extra_eng_cols)} features)',
                    f'Original\n({len(sym_cols)} features)'],
            autopct='%1.1f%%', colors=['#27AE60','#2E86C1'],
            textprops={'fontsize': 11})
axes[2].set_title("Feature Group Contribution to\nAvg Importance", fontsize=10)

plt.tight_layout(); plt.show()

# LDA discriminant loadings
lda_loadings = pd.DataFrame(np.abs(lda.coef_).mean(axis=0),
                              index=base_feat_cols[:len(lda.coef_[0])],
                              columns=['LDA_loading'])
print("\nTop 15 most discriminative features (LDA loadings):")
print(lda_loadings.sort_values('LDA_loading', ascending=False).head(15))


## 17. Confusion Matrix — Best Model

In [ ]:
best_model_name = results_df_sorted.index[0]
model_map = {
    "Random Forest"         : (best_rf,  X_test_rf),
    "XGBoost (LDA)"         : (best_xgb, X_test_xgb),
    "LightGBM (Hybrid)"     : (best_lgb, X_test_lgb),
    "CatBoost"              : (best_cat, X_test_cat),
    "Extra Trees"           : (best_et,  X_test_rf),
    "Voting (RF+LGB+Cat)"   : (voting_clf, X_test_hybrid),
    "Stacking (RF+ET+LGB → LR)": (stack_clf, X_test_rf),
}

best_m, best_X = model_map[best_model_name]
y_pred_best = best_m.predict(best_X)
class_names = le_main.inverse_transform(np.unique(y_test))

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, linewidths=0.3, cbar_kws={'shrink':0.6})
ax.set_title(f"Confusion Matrix — {best_model_name} (Test Set, 42 samples)",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("Actual", fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout(); plt.show()


## 18. Per-Class Classification Report — Best Model

In [ ]:
print(f"\n{'='*70}")
print(f"DETAILED CLASSIFICATION REPORT: {best_model_name}")
print(f"{'='*70}")
print(classification_report(y_test, y_pred_best,
                             target_names=class_names,
                             zero_division=0))

# Per-class accuracy bar chart
report_dict = {}
from sklearn.metrics import classification_report as cr
cr_text = cr(y_test, y_pred_best, target_names=class_names,
             zero_division=0, output_dict=True)
cr_df = pd.DataFrame(cr_text).T.iloc[:-3]   # drop avg rows

fig, ax = plt.subplots(figsize=(14, 9))
cr_df['f1-score'].sort_values().plot(kind='barh', ax=ax,
    color=['#E74C3C' if v < 0.5 else '#F39C12' if v < 0.8 else '#27AE60'
           for v in cr_df['f1-score'].sort_values()],
    edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', lw=1)
ax.set_xlabel("F1 Score"); ax.set_ylabel("Disease Class")
ax.set_title(f"Per-Class F1 Score — {best_model_name}", fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.1)
plt.tight_layout(); plt.show()


## 19. ROC-AUC Curves (OvR — Top 10 Classes)

In [ ]:
from sklearn.preprocessing import label_binarize

best_m_auc, best_X_auc = model_map[best_model_name]
y_score = best_m_auc.predict_proba(best_X_auc)
y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
n_classes  = y_test_bin.shape[1]

from sklearn.metrics import roc_curve, auc

# Plot top 10 classes by support
support = np.bincount(y_test)[:n_classes]
top10_idx = np.argsort(support)[-10:]

fig, ax = plt.subplots(figsize=(12, 8))
colors_roc = sns.color_palette("tab10", 10)

for i, (cls_idx, c) in enumerate(zip(top10_idx, colors_roc)):
    if y_test_bin[:, cls_idx].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_test_bin[:, cls_idx], y_score[:, cls_idx])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=1.8, color=c,
            label=f"{le_main.inverse_transform([cls_idx])[0][:30]} (AUC={roc_auc:.2f})")

ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title(f'ROC Curves (One-vs-Rest) — {best_model_name}\nTop 10 classes by support',
             fontsize=11, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 20. Dimensionality Reduction Impact Analysis

In [ ]:
# Compare same model (LightGBM) across different DR strategies
print("Comparing LightGBM across dimensionality reduction strategies...")
print("(Using best LightGBM params for fair comparison)\n")

lgb_params = best_lgb.get_params()

dr_comparison = {}
dr_configs = {
    'Raw 132 features'           : (train_fe[sym_cols].values, test_fe[sym_cols].values),
    'PCA (95%)'                  : (X_train_pca, X_test_pca),
    'LDA (40 components)'        : (X_train_lda, X_test_lda),
    'TruncatedSVD (50 comp)'     : (X_train_svd, X_test_svd),
    'Hybrid (LDA + Engineered)'  : (X_train_hybrid, X_test_hybrid),
}

for dr_name, (Xtr, Xte) in dr_configs.items():
    clf = lgb.LGBMClassifier(**lgb_params, verbose=-1)
    clf.fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf.predict(Xte))
    f1  = f1_score(y_test, clf.predict(Xte), average='weighted', zero_division=0)
    cv  = cross_val_score(clf, Xtr, y_train, cv=cv_strat, scoring='f1_weighted').mean()
    dr_comparison[dr_name] = {'n_features': Xtr.shape[1], 'Test_Acc': round(acc,4),
                               'Test_F1': round(f1,4), 'CV_F1': round(cv,4)}
    print(f"  {dr_name:35s} | features={Xtr.shape[1]:3d} | acc={acc:.4f} | f1={f1:.4f} | cv={cv:.4f}")

dr_comp_df = pd.DataFrame(dr_comparison).T
print("\n")
print(dr_comp_df.sort_values('Test_F1', ascending=False).to_string())

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Impact of Dimensionality Reduction Strategy on LightGBM Performance",
             fontsize=12, fontweight='bold')

dr_comp_df['Test_F1'].sort_values().plot(kind='barh', ax=axes[0],
    color=sns.color_palette("viridis", len(dr_comp_df)), edgecolor='white')
axes[0].set_xlabel("F1 Weighted (Test Set)")
axes[0].set_title("Test F1 by DR Strategy")
axes[0].axvline(0.9, color='red', linestyle='--', alpha=0.5)

axes[1].scatter(dr_comp_df['n_features'], dr_comp_df['Test_F1'],
                s=120, c=range(len(dr_comp_df)), cmap='viridis', zorder=5)
for name, row in dr_comp_df.iterrows():
    axes[1].annotate(name[:20], (row['n_features'], row['Test_F1']),
                     textcoords='offset points', xytext=(5,3), fontsize=8)
axes[1].set_xlabel("Number of Features"); axes[1].set_ylabel("F1 Weighted")
axes[1].set_title("Feature Count vs Performance")

plt.tight_layout(); plt.show()


## 21. Conclusion

### Key Findings

This notebook developed an advanced Medical Diagnostic System predicting 41 disease classes
from 132 binary symptoms. After deduplication, 304 samples × 132 features posed three
fundamental challenges — **multicollinearity** (63 highly correlated pairs), **extreme
dimensionality** (133 features for 304 samples), and **multinomial complexity** (41 classes).

**Multicollinearity Handling:**
- Hierarchical clustering grouped correlated symptoms → one representative per cluster
- Iterative VIF elimination (threshold=10) removed linearly redundant features
- Both methods together reduced original 132 features to a clean, orthogonal subset

**Dimensionality Reduction:**
- PCA (95% variance) → compact unsupervised representation
- **LDA (40 discriminant components)** — theoretically optimal for 41-class problem,
  maximises between-class separation in supervised manner
- TruncatedSVD → works directly on sparse binary matrices without centering
- Hybrid approach (LDA + domain-engineered features) outperformed all others

**Feature Engineering:**
- 10 clinical symptom group scores (gastro, neuro, respiratory, cardiac, etc.)
- Symptom rarity-weighted score (rare symptoms carry more diagnostic signal)
- Symptom entropy and system breadth (multi-organ involvement patterns)
- Chi² + Mutual Information filter selection as complementary filter methods

**Model Performance:**
Seven models compared — Random Forest, XGBoost (on LDA), LightGBM (Hybrid), CatBoost,
Extra Trees, Soft Voting, and Stacking. All used `class_weight='balanced'` to handle
the post-dedup imbalance. The Hybrid feature space (LDA + engineered) consistently
outperformed raw features, confirming that supervised dimensionality reduction
is the right choice for a 41-class multinomial target.

**Future Work:**
- Test with larger datasets (more patient rows per disease)
- Explore neural network approaches (TabNet, MLP with label smoothing)
- Add calibration (Platt scaling) for better probability estimates per class
- Consider hierarchical classification (disease family → specific disease)
